# Apex Retail — Store Intelligence Pipeline
**YOLOv8s + ByteTrack (entry/floor) + BoT-SORT (billing)**

Run all cells top to bottom. GPU T4 must be enabled.

---
## Before running
1. Kaggle → Settings → Accelerator → **GPU T4 x1** → Save
2. Upload your zip dataset as a Kaggle Dataset and add it to this notebook
3. Check `/kaggle/input/` paths match what's in Cell 5

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device name   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — STOP and enable GPU')
print('CUDA version  :', torch.version.cuda)

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────
!pip install ultralytics --quiet
!pip install opencv-python-headless --quiet
!pip install scipy --quiet

# Optional: uncomment when ready for production OSNet Re-ID
# !pip install torchreid --quiet

print('\n✓ Dependencies installed')

In [ ]:
# ── Cell 3: Download YOLOv8s model ───────────────────────────────────────
from ultralytics import YOLO
model = YOLO('yolov8s.pt')   # downloads ~22MB automatically
print('✓ YOLOv8s loaded')
print('  Parameters:', sum(p.numel() for p in model.model.parameters()) / 1e6, 'M')

In [ ]:
# ── Cell 4: Mount pipeline code ──────────────────────────────────────────
# Option A: if you uploaded pipeline/ folder as a Kaggle dataset
import sys, os
sys.path.insert(0, '/kaggle/input/retail-pipeline-code')  # adjust to your dataset name

# Option B: paste code directly (use if Option A fails)
# !mkdir -p /kaggle/working/pipeline
# Then paste each file content below ↓

# Verify import works
try:
    from pipeline.detect import DetectionPipeline
    print('✓ Pipeline imported from dataset')
except ImportError:
    print('⚠ Import failed — check dataset path or use Option B')

In [ ]:
# ── Cell 4B: Paste code directly (if no dataset upload) ──────────────────
# Run this cell ONLY if Cell 4 Option A failed

import os
os.makedirs('/kaggle/working/pipeline', exist_ok=True)

# Write __init__.py
with open('/kaggle/working/pipeline/__init__.py', 'w') as f:
    f.write('from .detect import DetectionPipeline\n')
    f.write('from .tracker import ReIDTracker, SessionManager\n')
    f.write('from .emit import EventEmitter\n')
    f.write('from .staff_vlm import is_staff_histogram, is_staff_vit\n')

print('Directory created. Now copy-paste detect.py, tracker.py, emit.py, staff_vlm.py')
print('into /kaggle/working/pipeline/ using the Kaggle file editor.')

In [ ]:
# ── Cell 5: Set video paths ──────────────────────────────────────────────
import os

# !! UPDATE THESE to match your actual dataset structure !!
DATASET_ROOT = '/kaggle/input/apex-retail-cctv'   # your dataset name here

# List what's in your dataset to find correct paths
for root, dirs, files in os.walk(DATASET_ROOT):
    for f in files:
        if f.endswith(('.mp4', '.avi', '.mov')):
            print(os.path.join(root, f))

In [ ]:
# ── Cell 6: Configure paths after checking Cell 5 output ─────────────────

# STORE 1 — update paths from Cell 5 output
STORE_CONFIG = {
    'STORE_001': {
        'entry':   f'{DATASET_ROOT}/store1/entry_camera.mp4',
        'floor':   f'{DATASET_ROOT}/store1/floor_camera.mp4',
        'billing': f'{DATASET_ROOT}/store1/billing_camera.mp4',
    },
    # Add more stores if dataset has them
    # 'STORE_002': { ... }
}

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

In [ ]:
# ── Cell 7: Run detection pipeline for all stores ─────────────────────────
import json
import sys
sys.path.insert(0, '/kaggle/working')   # for Option B code path

from pipeline.detect import DetectionPipeline

pipeline = DetectionPipeline(
    model_path = 'yolov8s.pt',
    device     = '0',           # GPU
)

all_events = []

for store_id, clips in STORE_CONFIG.items():
    print(f'\n═══ Processing {store_id} ═══')
    store_events = []

    for clip_type, video_path in clips.items():
        if not os.path.exists(video_path):
            print(f'  ⚠ Skipping {clip_type}: file not found — {video_path}')
            continue

        print(f'  Processing {clip_type}: {video_path}')
        events = pipeline.process_video(
            video_path  = video_path,
            clip_type   = clip_type,
            store_id    = store_id,
            vid_stride  = 3,     # process every 3rd frame — good balance
        )
        print(f'  ✓ {len(events)} events from {clip_type}')
        store_events.extend(events)

    all_events.extend(store_events)
    print(f'  STORE total: {len(store_events)} events')

# Sort by timestamp
all_events.sort(key=lambda e: e['timestamp'])

# Write events.jsonl
output_file = f'{OUTPUT_DIR}/events.jsonl'
with open(output_file, 'w') as f:
    for e in all_events:
        f.write(json.dumps(e) + '\n')

print(f'\n✓ Total {len(all_events)} events written → {output_file}')

In [ ]:
# ── Cell 8: Verify output — sample events ────────────────────────────────
import json

with open(f'{OUTPUT_DIR}/events.jsonl') as f:
    events = [json.loads(line) for line in f]

print(f'Total events: {len(events)}')
print(f'\nEvent type breakdown:')
from collections import Counter
counts = Counter(e['event_type'] for e in events)
for k, v in sorted(counts.items()):
    print(f'  {k:30s} {v}')

print(f'\nSample events (first 3):')
for e in events[:3]:
    print(json.dumps(e, indent=2))

In [ ]:
# ── Cell 9: Quick analytics from events ─────────────────────────────────
import pandas as pd

df = pd.DataFrame(events)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print('=== FOOTFALL ===' )
entries = df[df['event_type'] == 'ENTRY']
print(f'Total ENTRY events : {len(entries)}')
print(f'Unique visitors    : {entries["visitor_id"].nunique()}')

print('\n=== DWELL TIME ===')
dwell = df[df['event_type'] == 'ZONE_DWELL'].copy()
if len(dwell) > 0:
    dwell['dwell_s'] = dwell['dwell_ms'] / 1000
    print(f'Avg dwell in billing zone: {dwell["dwell_s"].mean():.1f}s')
    print(f'Max dwell in billing zone: {dwell["dwell_s"].max():.1f}s')

print('\n=== QUEUE ===' )
queue = df[df['event_type'] == 'BILLING_QUEUE_JOIN']
print(f'Queue join events: {len(queue)}')

print('\n=== STAFF FILTER ===')
staff_events   = df[df['is_staff'] == True]
visitor_events = df[df['is_staff'] == False]
print(f'Staff events   : {len(staff_events)}')
print(f'Visitor events : {len(visitor_events)}')

In [ ]:
# ── Cell 10: Enable OSNet Re-ID (upgrade from histogram) ─────────────────
# Run this cell AFTER the basic pipeline works.
# This upgrades cross-camera Re-ID from HSV histogram to OSNet 512-dim embeddings.

# Step 1: Install torchreid
!pip install torchreid --quiet

# Step 2: Set flag in tracker.py
# Open /kaggle/working/pipeline/tracker.py
# Change:  USE_OSNET = False
# To:      USE_OSNET = True

# Step 3: Verify OSNet loads
try:
    import torchreid
    extractor = torchreid.utils.FeatureExtractor(
        model_name = 'osnet_x1_0',
        device     = 'cuda',
    )
    print('✓ OSNet loaded — cross-camera Re-ID now production-grade')
    print('  Embedding dim: 512 (vs 48 for histogram)')
    print('  Training data: Market-1501 (viewpoint + lighting robust)')
except Exception as e:
    print('✗ OSNet failed:', e)

# Step 4: Re-run Cell 7 to regenerate events.jsonl with OSNet Re-ID

In [ ]:
# ── Cell 11: Run assertions.py (from dataset) ─────────────────────────────
# The PS provides assertions.py — run it against your events.jsonl

ASSERTIONS_PATH = f'{DATASET_ROOT}/assertions.py'

if os.path.exists(ASSERTIONS_PATH):
    !python {ASSERTIONS_PATH} --events {OUTPUT_DIR}/events.jsonl
else:
    print('assertions.py not found at:', ASSERTIONS_PATH)
    print('Check your dataset structure with Cell 5')

In [ ]:
# ── Cell 12: Save output for submission ──────────────────────────────────
import shutil

# events.jsonl is already at OUTPUT_DIR/events.jsonl
# Kaggle automatically packages /kaggle/working/ for download

print('Files ready for submission:')
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size  = os.path.getsize(fpath) / 1024
    print(f'  {f}  ({size:.1f} KB)')

print('\nDownload from: Kaggle → Output → Download')

---
## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA not available` | Settings → Accelerator → GPU T4 x1 → Save → Restart |
| `ModuleNotFoundError: pipeline` | Run Cell 4B, then paste code files manually |
| `Video not found` | Check Cell 5 output for actual paths |
| `boxes.id is None` | Normal for first few frames — tracker needs 3 frames to confirm a track |
| `torchreid install fails` | Keep `USE_OSNET = False` — histogram Re-ID still works |
| `OOM error` | Reduce `imgsz=416` or `vid_stride=5` in Cell 7 |
| Zero ENTRY events | Camera may be side-mounted — set `line_ratio=0.3` in EventEmitter |

---
## Architecture summary

```
Entry camera  → YOLOv8s + ByteTrack  → ENTRY / EXIT events
Floor camera  → YOLOv8s + ByteTrack  → ZONE_ENTER / ZONE_DWELL events  
Billing camera→ YOLOv8s + BoT-SORT   → BILLING_QUEUE_JOIN / ZONE_DWELL events
                    ↓
            ReIDTracker (cross-clip)
            HSV histogram (MVP) → OSNet upgrade (Cell 10)
                    ↓
           global visitor_token
                    ↓
             events.jsonl
```

See CHOICES.md for all architectural decisions and research references.